# Calculating Net Market Income 

In [3]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]


In [4]:
# Definimos los parámetros de la tabla ISR (Basado en tu código de Stata)
# Nota: En un proyecto real, podrías tener una tabla para 2018 y otra para 2024
TAX_BRACKETS = [
    {"li": 0.01,       "ls": 8952.49,    "cf": 0,          "tm": 0.0192},
    {"li": 8952.50,    "ls": 75984.55,   "cf": 171.88,     "tm": 0.064},
    {"li": 75984.56,   "ls": 133536.07,  "cf": 4461.94,    "tm": 0.1088},
    {"li": 133536.08,  "ls": 155229.80,  "cf": 10723.55,   "tm": 0.16},
    {"li": 155229.81,  "ls": 185852.57,  "cf": 14194.54,   "tm": 0.1792},
    {"li": 185852.58,  "ls": 374837.88,  "cf": 19682.13,   "tm": 0.2136},
    {"li": 374837.89,  "ls": 590795.99,  "cf": 60049.40,   "tm": 0.2352},
    {"li": 590796.00,  "ls": 1127926.84, "cf": 110842.74,  "tm": 0.30},
    {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99,  "tm": 0.32},
    {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17,  "tm": 0.34},
    {"li": 4511707.38, "ls": np.inf,     "cf": 1414947.85, "tm": 0.35},
]

In [5]:
def calculate_gross_income(df, brackets):
    """Calcula el ingreso bruto a partir del neto trimestral (Gross-up)"""
    df['ing_bruto'] = np.nan
    
    for b in brackets:
        # Aplicamos la fórmula inversa: Gross = (Net + CF - TM * LI) / (1 - TM)
        b_temp = (df['ing_tri_anual'] + b['cf'] - b['tm'] * b['li']) / (1 - b['tm'])
        
        # Máscara para identificar qué registros caen en este tramo
        mask = (df['ing_bruto'].isna()) & \
               (b_temp >= b['li']) & \
               (b_temp <= b['ls'])
        
        df.loc[mask, 'ing_bruto'] = b_temp
    
    return df

# Ciclo para procesar ambos años
for year in YEARS:
    print(f"📊 Procesando año {year}...")
    
    # 1. Cargar datos de ingresos
    file_path = RAW_DIR / f"ingresos_{year}.csv"
    df_ing = pd.read_csv(file_path)
    
    # 2. Filtrar por clave P001 (Sueldos y salarios)
    df_p001 = df_ing[df_ing['clave'] == 'P001'].copy()
    
    # 3. Aplicar el cálculo del ingreso bruto
    # Asegúrate de que 'ing_tri' es el nombre de la columna en tu CSV
    df_p001 = df_p001.rename(columns={'ing_tri': 'ing_tri_anual'}) 
    df_p001 = calculate_gross_income(df_p001, TAX_BRACKETS)
    
    # 4. Guardar avance
    output_path = PROCESSED_DIR / f"ingresos_brutos_{year}.csv"
    df_p001.to_csv(output_path, index=False)
    print(f"   ✅ Guardado en: {output_path.name}")

print("\n✨ Primer paso del CEQ completado.")

📊 Procesando año 2018...
   ✅ Guardado en: ingresos_brutos_2018.csv
📊 Procesando año 2024...
   ✅ Guardado en: ingresos_brutos_2024.csv

✨ Primer paso del CEQ completado.


In [8]:
# 1. Cargar los DataFrames procesados
df_2018 = pd.read_csv(PROCESSED_DIR / "ingresos_brutos_2018.csv")
df_2024 = pd.read_csv(PROCESSED_DIR / "ingresos_brutos_2024.csv")

# 2. Mostrar un resumen rápido de 2018
print(f"{'='*20} ENIGH 2018 {'='*20}")
print(f"Registros procesados (P001): {len(df_2018)}")
display(df_2018[['folioviv', 'foliohog', 'numren', 'ing_tri_anual', 'ing_bruto']].head())

# 3. Mostrar un resumen rápido de 2024
print(f"\n{'='*20} ENIGH 2024 {'='*20}")
print(f"Registros procesados (P001): {len(df_2024)}")
display(df_2024[['folioviv', 'foliohog', 'numren', 'ing_tri_anual', 'ing_bruto']].head())

# 4. Comparación estadística rápida de la columna generada
summary_comparison = pd.concat([
    df_2018['ing_bruto'].describe().rename('2018'),
    df_2024['ing_bruto'].describe().rename('2024')
], axis=1)

print("\n--- Resumen Estadístico del Ingreso Bruto (Market Income) ---")
display(summary_comparison)

==================== ENIGH 2018 ====================
Registros procesados (P001): 88886


,folioviv,foliohog,numren,ing_tri_anual,ing_bruto
0,100013601,1,1,29508.19,31097.339744
1,100013601,1,3,23606.55,24792.168803
2,100013603,1,1,110163.93,119343.301023
3,100013603,1,2,23606.55,24792.168803
4,100013606,1,3,8852.45,9029.241453



==================== ENIGH 2024 ====================
Registros procesados (P001): 106397


,folioviv,foliohog,numren,ing_tri_anual,ing_bruto
0,100001901,1,1,48952.17,51870.822650
1,100001901,1,2,29347.82,30926.004274
2,100001902,1,1,46956.52,49738.717949
3,100001902,1,3,29347.82,30926.004274
4,100001904,1,1,5869.56,5984.461468



--- Resumen Estadístico del Ingreso Bruto (Market Income) ---


,2018,2024
count,8.888600e+04,1.063970e+05
mean,1.864511e+04,3.025013e+04
std,2.044863e+04,2.826258e+04
min,1.245902e+01,2.492843e+01
25%,8.424031e+03,1.587583e+04
50%,1.487111e+04,2.479217e+04
75%,2.277384e+04,3.719691e+04
max,1.279503e+06,1.576070e+06


# Imputation of IVA

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración de prefijos IVA y códigos de informalidad por año
YEAR_CONFIG = {
    2018: {
        # Prefijos: C (Limpieza), D (Cuidados), F (Comunicaciones/Combustible), 
        # H (Vestido), I (Muebles), K (Enseres), L (Esparcimiento), M (Transporte), N (Otros)
        "iva_prefixes": ('C', 'D', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N'),
        "informal_locs": [1, 2, 3, 17], # Códigos 2018
        "clave_len": 4
    },
    2024: {
        # Prefijos COICOP: 02 (Alcohol), 03 (Vestido), 05 (Muebles), 07 (Transporte), 
        # 08 (Comunicación), 09 (Recreación), 11 (Restaurantes), 12 (Otros)
        "iva_prefixes": ('012', '02', '03', '05', '07', '08', '09', '11', '12'),
        "informal_locs": [1, 2, 3, 17], # Generalmente se mantienen, verificar en FD
        "clave_len": 6
    }
}

def fix_and_calculate_iva(year, raw_dir):
    print(f"🛠️  Procesando ENIGH {year}...")
    
    config = YEAR_CONFIG[year]
    
    # 1. Carga de datos
    df_gastos = pd.read_csv(raw_dir / f"gastoshogar_{year}.csv", 
                            dtype={'folioviv': str, 'foliohog': str, 'clave': str}, 
                            low_memory=False)
    df_conc = pd.read_csv(raw_dir / f"concentradohogar_{year}.csv", 
                          dtype={'folioviv': str, 'foliohog': str}, 
                          low_memory=False)

    # 2. Procesamiento de Gastos
    df_gastos['gasto_tri'] = pd.to_numeric(df_gastos['gasto_tri'], errors='coerce').fillna(0)
    
    # Filtrar solo gastos monetarios (G1)
    df_gastos = df_gastos[df_gastos['tipo_gasto'] == 'G1'].copy()
    
    # Identificar si lleva IVA basado en los prefijos del año correspondiente
    df_gastos['lleva_iva'] = df_gastos['clave'].str.startswith(config["iva_prefixes"]).astype(int)
    
    # Identificar formalidad (ajustar si los códigos de lugar_comp variaron)
    df_gastos['es_formal'] = (~df_gastos['lugar_comp'].isin(config["informal_locs"])).astype(int)
    
    # Cálculo del IVA por partida: Gasto * (0.16 / 1.16)
    # $Factor_{IVA} = \frac{0.16}{1.16} \approx 0.137931$
    factor_iva = 0.137931034
    df_gastos['iva_partida'] = np.where((df_gastos['lleva_iva'] == 1) & (df_gastos['es_formal'] == 1),
                                        df_gastos['gasto_tri'] * factor_iva, 0)
    
    # Agrupar IVA por hogar
    iva_hogar = df_gastos.groupby(['folioviv', 'foliohog'])['iva_partida'].sum().reset_index()
    iva_hogar.rename(columns={'iva_partida': 'IVA_HH'}, inplace=True)

    # 3. Preparar Concentrado
    cols_num = ['gasto_mon', 'alimentos', 'limpieza', 'cuidados', 'ing_cor', 'factor']
    for col in cols_num:
        df_conc[col] = pd.to_numeric(df_conc[col], errors='coerce').fillna(0)
    
    # Base de gasto estructural (excluyendo lo básico que suele ser tasa 0% o exento)
    df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])

    # 4. Merge y cálculo de tasa efectiva
    df_final = pd.merge(df_conc, iva_hogar, on=['folioviv', 'foliohog'], how='left').fillna(0)
    
    # Cálculo del IVA atribuible al ingreso (proporcional)
    # Evitamos división por cero con np.where
    df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,
                                    (df_final['IVA_HH'] / df_final['Gasto_HH_tri']) * df_final['ing_cor'],
                                    0)
    
    return df_final.copy()

# Ejecución del Loop
results = {}
for year in [2018, 2024]:
    # Asumiendo que RAW_DIR está definido previamente como un objeto Path
    df_result = fix_and_calculate_iva(year, RAW_DIR)
    
    # Cálculo de Deciles
    df_result = df_result.sort_values("ing_cor")
    df_result["cumsum_factor"] = df_result["factor"].cumsum()
    df_result["decil"] = pd.qcut(df_result["cumsum_factor"], 10, labels=range(1, 11))
    
    # Resumen ponderado
    summary = df_result.groupby("decil").agg(
        ingreso_medio=("ing_cor", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        iva_medio=("IVA_HH_A", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        hogares_representados=("factor", "sum")
    ).reset_index()
    
    summary["carga_iva_pct"] = (summary["iva_medio"] / summary["ingreso_medio"]) * 100
    results[year] = summary

    print(f"\n{'='*20} ENIGH {year} {'='*20}")
    display(summary.style.format({
        "ingreso_medio": "${:,.2f}", "iva_medio": "${:,.2f}", 
        "carga_iva_pct": "{:.2f}%", "hogares_representados": "{:,.0f}"
    }))

🛠️  Procesando ENIGH 2018...

==================== ENIGH 2018 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3363239983.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3363239983.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$8,716.22","$1,658.35","3,077,839",19.03%
1,2,"$15,291.60","$2,251.26","3,127,662",14.72%
2,3,"$20,324.76","$2,223.45","3,221,401",10.94%
3,4,"$25,314.82","$2,806.68","3,299,920",11.09%
4,5,"$30,614.86","$3,098.35","3,412,828",10.12%
5,6,"$36,961.89","$3,629.89","3,460,309",9.82%
6,7,"$44,727.99","$4,388.86","3,501,973",9.81%
7,8,"$55,585.77","$5,446.05","3,575,937",9.80%
8,9,"$73,631.94","$8,816.83","3,682,880",11.97%
9,10,"$156,510.85","$18,790.49","4,039,766",12.01%


🛠️  Procesando ENIGH 2024...

==================== ENIGH 2024 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3363239983.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3363239983.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$15,688.53","$5,014.86","3,283,413",31.97%
1,2,"$26,482.16","$3,675.52","3,532,617",13.88%
2,3,"$34,539.90","$4,416.71","3,634,153",12.79%
3,4,"$42,452.03","$5,475.26","3,746,813",12.90%
4,5,"$51,059.62","$6,283.88","3,861,828",12.31%
5,6,"$60,895.15","$7,609.87","3,949,553",12.50%
6,7,"$72,846.22","$9,077.25","3,922,847",12.46%
7,8,"$89,292.18","$12,463.10","4,055,830",13.96%
8,9,"$115,561.16","$14,573.56","4,229,375",12.61%
9,10,"$220,959.36","$34,181.70","4,613,801",15.47%


In [16]:
import pandas as pd
import numpy as np

def fix_and_calculate_iva(year):
    print(f"\n🛠️  Corrigiendo y procesando {year}...")
    
    # 1. Cargar datos
    df_gastos = pd.read_csv(RAW_DIR / f"gastoshogar_{year}.csv",
                            dtype={'folioviv': str, 'foliohog': str},
                            low_memory=False)
    
    df_conc = pd.read_csv(RAW_DIR / f"concentradohogar_{year}.csv",
                          dtype={'folioviv': str, 'foliohog': str},
                          low_memory=False)

    # 2. Limpieza básica
    df_gastos['gasto_tri'] = pd.to_numeric(df_gastos['gasto_tri'], errors='coerce').fillna(0)

    print("\n--- DIAGNÓSTICO INICIAL ---")
    print("Total registros (gastos):", len(df_gastos))
    print("Tipos de gasto:", df_gastos['tipo_gasto'].value_counts().head())

    # 3. Filtrar tipo de gasto
    df_gastos = df_gastos[df_gastos['tipo_gasto'] == 'G1'].copy()
    print("Registros después de filtrar G1:", len(df_gastos))

    # 4. Revisar claves
    df_gastos['clave'] = df_gastos['clave'].astype(str)
    print("\nTop claves (primeros 3 dígitos):")
    print(df_gastos['clave'].str[:3].value_counts().head(10))

    iva_prefixes = ('012', '02', '03', '05', '07', '08', '09', '11')
    df_gastos['lleva_iva'] = df_gastos['clave'].str.startswith(iva_prefixes).astype(int)

    print("Con IVA potencial:", df_gastos['lleva_iva'].sum())

    # 5. Revisar lugar de compra
    print("\nDistribución lugar_comp:")
    print(df_gastos['lugar_comp'].value_counts().head(10))

    informal_locs = [1, 2, 3, 17]
    df_gastos['lugar_comp'] = pd.to_numeric(df_gastos['lugar_comp'], errors='coerce')
    df_gastos['es_formal'] = (~df_gastos['lugar_comp'].isin(informal_locs)).astype(int)

    print("Compras formales:", df_gastos['es_formal'].sum())

    # 6. Calcular IVA
    factor_iva = 0.137931034
    df_gastos['iva_partida'] = np.where(
        (df_gastos['lleva_iva'] == 1) & (df_gastos['es_formal'] == 1),
        df_gastos['gasto_tri'] * factor_iva,
        0
    )

    print("Registros con IVA positivo:", (df_gastos['iva_partida'] > 0).sum())

    # 7. Agrupar IVA por hogar
    iva_hogar = df_gastos.groupby(['folioviv', 'foliohog'])['iva_partida'].sum().reset_index()
    iva_hogar.rename(columns={'iva_partida': 'IVA_HH'}, inplace=True)

    print("Hogares con IVA:", (iva_hogar['IVA_HH'] > 0).sum())

    # 8. Preparar concentrado
    cols_num = ['gasto_mon', 'alimentos', 'limpieza', 'cuidados', 'ing_cor', 'factor']
    for col in cols_num:
        df_conc[col] = pd.to_numeric(df_conc[col], errors='coerce').fillna(0)

    df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (
        df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados']
    )

    # 9. Merge
    df_final = pd.merge(df_conc, iva_hogar,
                        on=['folioviv', 'foliohog'],
                        how='left').fillna(0)

    print("Hogares totales:", len(df_final))
    print("Hogares con IVA después del merge:", (df_final['IVA_HH'] > 0).sum())

    # 10. Calcular IVA anual
    df_final['IVA_HH_A'] = np.where(
        df_final['Gasto_HH_tri'] > 0,
        (df_final['IVA_HH'] / df_final['Gasto_HH_tri']) * df_final['ing_cor'],
        0
    )

    return df_final.copy()


# -------------------------
# EJECUCIÓN
# -------------------------
for year in [2018, 2024]:
    df_result = fix_and_calculate_iva(year)

    # Deciles
    df_result = df_result.sort_values("ing_cor")
    df_result["cumsum_factor"] = df_result["factor"].cumsum()
    df_result["decil"] = pd.qcut(df_result["cumsum_factor"], 10, labels=range(1, 11))

    # Resumen
    summary = df_result.groupby("decil").agg(
        ingreso_medio=("ing_cor", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        iva_medio=("IVA_HH_A", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        hogares_representados=("factor", "sum")
    ).reset_index()

    summary["carga_iva_pct"] = (summary["iva_medio"] / summary["ingreso_medio"]) * 100

    print(f"\n{'='*20} TABLA {year} {'='*20}")
    print(summary)


🛠️  Corrigiendo y procesando 2018...

--- DIAGNÓSTICO INICIAL ---
Total registros (gastos): 4405250
Tipos de gasto: tipo_gasto
G1    4000205
G5     201451
G3      76714
G7      64308
G6      49892
Name: count, dtype: int64
Registros después de filtrar G1: 4000205

Top claves (primeros 3 dígitos):
clave
C00    371196
A00    316679
D00    295450
D01    200901
R00    181567
A01    164219
A12    130377
A22    127352
A24    124282
A09    122371
Name: count, dtype: int64
Con IVA potencial: 0

Distribución lugar_comp:
lugar_comp
4     1162716
6      801419
5      754057
0      427231
1      173682
3      154127
2      146363
17     106883
15      60573
12      56025
Name: count, dtype: int64
Compras formales: 3419150
Registros con IVA positivo: 0
Hogares con IVA: 0
Hogares totales: 74647
Hogares con IVA después del merge: 0

==================== TABLA 2018 ====================
  decil  ingreso_medio  iva_medio  hogares_representados  carga_iva_pct
0     1    8716.224214        0.0           

/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3157925653.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3157925653.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(



--- DIAGNÓSTICO INICIAL ---
Total registros (gastos): 5311497
Tipos de gasto: tipo_gasto
G1    4865922
G5     217915
G3      83751
G7      78758
G6      47719
Name: count, dtype: int64
Registros después de filtrar G1: 4865922

Top claves (primeros 3 dígitos):
clave
011    1658647
131     647600
056     585585
012     237843
031     236926
111     196646
083     168038
045     140002
181     126357
061     110643
Name: count, dtype: int64
Con IVA potencial: 1875923

Distribución lugar_comp:
lugar_comp
4     1252340
6     1010218
5      977965
0      611453
1      192047
3      170102
2      162579
17     136202
7       72292
12      63563
Name: count, dtype: int64
Compras formales: 4204992
Registros con IVA positivo: 1613941
Hogares con IVA: 91221
Hogares totales: 91414
Hogares con IVA después del merge: 91221

==================== TABLA 2024 ====================
  decil  ingreso_medio     iva_medio  hogares_representados  carga_iva_pct
0     1   15688.531537   5011.472389             

/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3157925653.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_1482/3157925653.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(


In [10]:
pd.read_csv(PROCESSED_DIR / "iva_imputado_2018.csv")

,folioviv,foliohog,ubica_geo,tam_loc,est_socio,est_dis,upm,factor,clase_hog,sexo_jefe,...,deposito,prest_terc,pago_tarje,deudas,balance,otras_erog,smg,Gasto_HH_tri,IVA_HH,IVA_HH_A
0,100013601,1,1001,1,3,2,1,179,2,1,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,11889.00,0.0,0.0
1,100013602,1,1001,1,3,2,1,179,2,1,...,9.83,0.0,8852.45,0.00,0.0,0.0,7952.4,33753.54,0.0,0.0
2,100013603,1,1001,1,3,2,1,179,2,1,...,66393.44,0.0,0.00,14754.09,0.0,0.0,7952.4,59854.57,0.0,0.0
3,100013604,1,1001,1,3,2,1,179,2,2,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,5419.22,0.0,0.0
4,100013606,1,1001,1,3,2,1,179,2,2,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,0.00,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74642,3260798902,1,32046,4,2,543,8395,192,2,1,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,9568.24,0.0,0.0
74643,3260798903,1,32046,4,2,543,8395,192,2,1,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,5033.66,0.0,0.0
74644,3260798904,1,32046,4,2,543,8395,192,3,1,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,23788.27,0.0,0.0
74645,3260798905,1,32046,4,2,543,8395,192,1,1,...,0.00,0.0,0.00,0.00,0.0,0.0,7952.4,6834.90,0.0,0.0


In [11]:
pd.read_csv(PROCESSED_DIR / "iva_imputado_2024.csv")

,folioviv,foliohog,ubica_geo,tam_loc,est_socio,est_dis,upm,factor,clase_hog,sexo_jefe,...,deposito,prest_terc,pago_tarje,deudas,balance,otras_erog,smg,Gasto_HH_tri,IVA_HH,IVA_HH_A
0,100001901,1,1001,1,3,1,1,207,2,1,...,21365.21,0.00,0.00,8217.39,0.00,0.00,22403.7,21996.33,3619.042746,22743.289089
1,100001902,1,1001,1,3,1,1,207,2,1,...,0.00,0.00,0.00,0.00,0.00,0.00,22403.7,13123.83,2938.166886,26421.017679
2,100001904,1,1001,1,3,1,1,207,2,2,...,4695.65,0.00,0.00,0.00,0.00,0.00,22403.7,17284.97,1560.532408,4231.214241
3,100001905,1,1001,1,3,1,1,207,2,1,...,5869.56,0.00,7336.95,2934.78,0.00,6260.86,22403.7,17458.18,2947.617921,18644.884046
4,100002501,1,1001,1,2,2,2,196,2,2,...,0.00,2934.78,0.00,0.00,0.00,7113.91,22403.7,80164.41,7765.278594,9637.687849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91409,3260593814,1,32052,4,2,680,10593,183,2,1,...,0.00,0.00,0.00,0.00,0.00,0.00,22403.7,6510.00,1705.279994,10507.107568
91410,3260593815,1,32052,4,2,680,10593,183,2,1,...,0.00,0.00,0.00,0.00,0.00,0.00,22403.7,21785.10,3668.566884,14768.221821
91411,3260593816,1,32052,4,2,680,10593,183,2,1,...,2950.81,0.00,0.00,0.00,0.00,0.00,22403.7,10522.22,1090.344824,5228.715856
91412,3260593817,1,32052,4,2,680,10593,183,2,1,...,0.00,0.00,0.00,0.00,10081.96,0.00,22403.7,7217.74,803.291032,1373.841861
